
# 12 — Sensitivity Analysis  
## Final 5-Variable POSet Robustness Synthesis

This notebook consolidates the sensitivity and robustness checks around the main POSet result.

It runs after:

```text
11_Recovery_Validation_2002_5Var.ipynb
```

## Main result being tested

The main result is the **5-variable, 5-level profile POSet** from Notebook 08.

## Final decisions reflected here

- Final ordering variables = **5**.
- WGI/governance is not part of final ordering.
- Recovery is validation only, not ordering.
- Volatility is diagnostic/sensitivity only, not ordering.
- Main POSet uses two single pre-shock cross-sections:
  - 2007
  - 2019

## Sensitivity checks synthesized here

1. **Discretization levels**
   - 3 levels
   - 4 levels
   - 5 levels

2. **Variable set**
   - baseline 5 variables
   - drop energy security as a sensitivity check

3. **Epsilon tolerance**
   - from Notebook 09
   - includes validity/cycle diagnostics

4. **Epsilon-margin robustness**
   - from Notebook 10
   - includes explicit epsilon grid and reference epsilon = 0.10

5. **Recovery validation**
   - from Notebook 11
   - descriptive validation only, no causal claim

## Main outputs

- `sensitivity_profile_discretization_main_set.csv`
- `sensitivity_variable_set_profile_summary.csv`
- `sensitivity_epsilon_tolerance_main_set.csv`
- `sensitivity_epsilon_margin_main_set.csv`
- `sensitivity_recovery_validation_main_set.csv`
- `robustness_dashboard_table.csv`
- `sensitivity_synthesis.csv`
- `report_table_profile_sensitivity.csv`
- `report_table_epsilon_margin.csv`
- `report_table_recovery_validation.csv`
- `report_ready_sensitivity_paragraphs.csv`


In [1]:

# ------------------------------------------------------
# IMPORTS AND PATHS
# ------------------------------------------------------

from pathlib import Path
from datetime import datetime
import itertools
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 300)
pd.set_option("display.float_format", "{:.4f}".format)

PROJECT_ROOT = Path.cwd()

MASTER_DIR = PROJECT_ROOT / "Data" / "Processed" / "Master_Dataset"
PRE_POSET_DIR = PROJECT_ROOT / "Data" / "Processed" / "Pre_POSet_EDA"
PROFILE_DIR = PROJECT_ROOT / "Data" / "Processed" / "Profile_POSet_Main"
EPSILON_TOLERANCE_DIR = PROJECT_ROOT / "Data" / "Processed" / "Epsilon_Sensitivity_Country_Level"
EPSILON_MARGIN_DIR = PROJECT_ROOT / "Data" / "Processed" / "Epsilon_Margin_POSet_Robustness"
RECOVERY_VALIDATION_DIR = PROJECT_ROOT / "Data" / "Processed" / "Recovery_Validation"

PROCESSED_DIR = PROJECT_ROOT / "Data" / "Processed" / "Sensitivity_Analysis"
DIAGNOSTICS_DIR = PROJECT_ROOT / "Data" / "Diagnostics" / "12_Sensitivity_Analysis"
TABLES_DIR = PROJECT_ROOT / "Outputs" / "Tables" / "Sensitivity_Analysis"
AUDIT_DIR = PROJECT_ROOT / "Data" / "Audit"

for folder in [PROCESSED_DIR, DIAGNOSTICS_DIR, TABLES_DIR, AUDIT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print("Run ID:", RUN_ID)
print("Project root:", PROJECT_ROOT.resolve())
print("Processed folder:", PROCESSED_DIR.resolve())


Run ID: 20260624_203131
Project root: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24
Processed folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\Sensitivity_Analysis


In [2]:

# ------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------

STRUCTURAL_COMPLETE_CASE_FILE = MASTER_DIR / "structural_snapshot_final5_complete_cases.csv"
PROFILE_QUALITY_FILE = PROFILE_DIR / "profile_poset_quality_diagnostics.csv"
DISCRETIZATION_DETAIL_FILE = PRE_POSET_DIR / "profile_discretization_detail.csv"
DISCRETIZATION_DIAGNOSTIC_FILE = PRE_POSET_DIR / "discretization_level_diagnostic.csv"
EPSILON_TOLERANCE_SUMMARY_FILE = EPSILON_TOLERANCE_DIR / "epsilon_run_summary.csv"
EPSILON_TOLERANCE_VALIDITY_FILE = EPSILON_TOLERANCE_DIR / "epsilon_validity_diagnostics.csv"
EPSILON_MARGIN_SUMMARY_FILE = EPSILON_MARGIN_DIR / "epsilon_margin_run_summary.csv"
EPSILON_MARGIN_VALIDITY_FILE = EPSILON_MARGIN_DIR / "epsilon_margin_validity_diagnostics.csv"
RECOVERY_TAKEAWAYS_FILE = RECOVERY_VALIDATION_DIR / "recovery_validation_takeaway_table.csv"
RECOVERY_FRONTIER_COMPARISON_FILE = RECOVERY_VALIDATION_DIR / "profile_frontier_recovery_comparison.csv"

REQUIRED_FILES = [
    STRUCTURAL_COMPLETE_CASE_FILE,
    PROFILE_QUALITY_FILE,
    DISCRETIZATION_DIAGNOSTIC_FILE,
    EPSILON_TOLERANCE_SUMMARY_FILE,
    EPSILON_MARGIN_SUMMARY_FILE,
    RECOVERY_TAKEAWAYS_FILE,
]

for path in REQUIRED_FILES:
    if not path.exists():
        raise FileNotFoundError(f"Required input missing: {path}")

FINAL_5_SCORE_COLUMNS = [
    "debt_capacity_score_0_100",
    "employment_strength_score_0_100",
    "rd_intensity_score_0_100",
    "human_capital_tertiary_score_0_100",
    "energy_security_score_0_100",
]

VARIABLE_SET_DEFINITIONS = {
    "baseline_5_variables": FINAL_5_SCORE_COLUMNS,
    "sensitivity_drop_energy_security": [
        "debt_capacity_score_0_100",
        "employment_strength_score_0_100",
        "rd_intensity_score_0_100",
        "human_capital_tertiary_score_0_100",
    ],
}

MAIN_VARIABLE_SET = "baseline_5_variables"
MAIN_LEVELS = 5
REFERENCE_EPSILON = 0.10

print("Main variable set:", MAIN_VARIABLE_SET)
print("Main levels:", MAIN_LEVELS)
print("Variable sets:")
for k, v in VARIABLE_SET_DEFINITIONS.items():
    print("-", k, len(v), "variables")


Main variable set: baseline_5_variables
Main levels: 5
Variable sets:
- baseline_5_variables 5 variables
- sensitivity_drop_energy_security 4 variables


In [3]:

# ------------------------------------------------------
# SAVE HELPER
# ------------------------------------------------------

table_inventory_rows = []

def save_table(table, file_name, title, description):
    processed_path = PROCESSED_DIR / file_name
    diagnostics_path = DIAGNOSTICS_DIR / file_name
    table_path = TABLES_DIR / file_name

    table.to_csv(processed_path, index=False)
    table.to_csv(diagnostics_path, index=False)
    table.to_csv(table_path, index=False)

    table_inventory_rows.append({
        "file_name": file_name,
        "title": title,
        "description": description,
        "rows": len(table),
        "columns": len(table.columns),
        "processed_path": str(processed_path),
        "diagnostics_path": str(diagnostics_path),
        "table_path": str(table_path),
        "created_at": RUN_TIMESTAMP,
    })

    print("Saved:", file_name)


In [4]:

# ------------------------------------------------------
# CORE POSET HELPER FUNCTIONS
# ------------------------------------------------------

def assign_quantile_levels(series, levels):
    ranked = series.rank(method="first")
    return pd.qcut(ranked, q=levels, labels=list(range(1, levels + 1))).astype(int)


def dominates(levels_a, levels_b):
    a = np.array(levels_a, dtype=int)
    b = np.array(levels_b, dtype=int)
    return bool(np.all(a >= b) and np.any(a > b))


def compute_profile_poset_summary(country_df, score_columns, levels, variable_set_name):
    level_cols = []

    df = country_df.copy()

    for col in score_columns:
        level_col = col.replace("_score_0_100", f"_level_{levels}")
        df[level_col] = assign_quantile_levels(df[col], levels)
        level_cols.append(level_col)

    df["profile_code"] = df[level_cols].astype(str).agg("-".join, axis=1)
    df["profile_digit_sum"] = df[level_cols].sum(axis=1)

    profile_nodes = (
        df.groupby(["shock_id", "baseline_year", "profile_code"] + level_cols)
        .agg(
            countries=("Country", lambda x: ", ".join(sorted(x))),
            country_count=("Country", "nunique"),
            profile_digit_sum=("profile_digit_sum", "first"),
        )
        .reset_index()
    )

    quality_rows = []

    for shock_id, nodes in profile_nodes.groupby("shock_id"):
        nodes = nodes.copy()
        baseline_year = int(nodes["baseline_year"].iloc[0])
        node_records = nodes.to_dict("records")

        dominance_pairs = []

        for a in node_records:
            a_levels = [a[c] for c in level_cols]

            for b in node_records:
                if a["profile_code"] == b["profile_code"]:
                    continue

                b_levels = [b[c] for c in level_cols]

                if dominates(a_levels, b_levels):
                    dominance_pairs.append((a["profile_code"], b["profile_code"]))

        dominance_pairs = set(dominance_pairs)

        # Hasse edges via transitive reduction.
        hasse_pairs = []

        for a, b in dominance_pairs:
            transitive = False

            all_profiles = set(nodes["profile_code"])

            for c in all_profiles:
                if c == a or c == b:
                    continue
                if (a, c) in dominance_pairs and (c, b) in dominance_pairs:
                    transitive = True
                    break

            if not transitive:
                hasse_pairs.append((a, b))

        profiles = sorted(nodes["profile_code"].unique())
        total_pairs = 0
        comparable_pairs = 0

        for a, b in itertools.combinations(profiles, 2):
            total_pairs += 1
            if (a, b) in dominance_pairs or (b, a) in dominance_pairs:
                comparable_pairs += 1

        dominated_profiles = set([b for _, b in dominance_pairs])
        frontier_profiles = [p for p in profiles if p not in dominated_profiles]

        frontier_countries = nodes[nodes["profile_code"].isin(frontier_profiles)]["country_count"].sum()

        # Layers.
        remaining = set(profiles)
        layer = 1
        max_layer = 0

        while remaining:
            layer_profiles = []
            for profile in sorted(remaining):
                dominated_by_remaining = any((other, profile) in dominance_pairs for other in remaining if other != profile)
                if not dominated_by_remaining:
                    layer_profiles.append(profile)

            if not layer_profiles:
                max_layer = np.nan
                break

            remaining -= set(layer_profiles)
            max_layer = layer
            layer += 1

        n_countries = int(nodes["country_count"].sum())
        n_profiles = int(nodes["profile_code"].nunique())

        quality_rows.append({
            "shock_id": shock_id,
            "baseline_year": baseline_year,
            "variable_set": variable_set_name,
            "n_variables": len(score_columns),
            "levels": levels,
            "countries": n_countries,
            "unique_profiles": n_profiles,
            "profile_uniqueness_ratio": n_profiles / n_countries if n_countries else np.nan,
            "dominance_relations": len(dominance_pairs),
            "hasse_edges": len(hasse_pairs),
            "layers": max_layer,
            "frontier_profiles": len(frontier_profiles),
            "frontier_countries": int(frontier_countries),
            "comparability_ratio": comparable_pairs / total_pairs if total_pairs else np.nan,
            "incomparability_ratio": (total_pairs - comparable_pairs) / total_pairs if total_pairs else np.nan,
        })

    return pd.DataFrame(quality_rows)


In [5]:

# ------------------------------------------------------
# LOAD INPUT TABLES
# ------------------------------------------------------

structural = pd.read_csv(STRUCTURAL_COMPLETE_CASE_FILE)
profile_quality_main = pd.read_csv(PROFILE_QUALITY_FILE)
discretization_diagnostic = pd.read_csv(DISCRETIZATION_DIAGNOSTIC_FILE)
epsilon_tolerance_summary = pd.read_csv(EPSILON_TOLERANCE_SUMMARY_FILE)
epsilon_margin_summary = pd.read_csv(EPSILON_MARGIN_SUMMARY_FILE)
recovery_takeaways = pd.read_csv(RECOVERY_TAKEAWAYS_FILE)

if EPSILON_TOLERANCE_VALIDITY_FILE.exists():
    epsilon_tolerance_validity = pd.read_csv(EPSILON_TOLERANCE_VALIDITY_FILE)
else:
    epsilon_tolerance_validity = pd.DataFrame()

if EPSILON_MARGIN_VALIDITY_FILE.exists():
    epsilon_margin_validity = pd.read_csv(EPSILON_MARGIN_VALIDITY_FILE)
else:
    epsilon_margin_validity = pd.DataFrame()

if RECOVERY_FRONTIER_COMPARISON_FILE.exists():
    recovery_frontier_comparison = pd.read_csv(RECOVERY_FRONTIER_COMPARISON_FILE)
else:
    recovery_frontier_comparison = pd.DataFrame()

for df_ in [structural, profile_quality_main, discretization_diagnostic, epsilon_tolerance_summary, epsilon_margin_summary, recovery_takeaways]:
    if "shock_id" in df_.columns:
        df_["shock_id"] = df_["shock_id"].astype(str)

structural["Country"] = structural["Country"].astype(str).str.strip().str.upper()
structural["shock_id"] = structural["shock_id"].astype(str)

for col in FINAL_5_SCORE_COLUMNS:
    if col not in structural.columns:
        raise ValueError(f"Missing final 5 score column in structural data: {col}")
    structural[col] = pd.to_numeric(structural[col], errors="coerce")

structural = structural.dropna(subset=FINAL_5_SCORE_COLUMNS).copy()

print("Loaded structural rows:", len(structural))
display(structural.groupby(["shock_id", "baseline_year"])["Country"].nunique().reset_index(name="countries"))


Loaded structural rows: 60


,shock_id,baseline_year,countries
0,2007,2007,25
1,2019,2019,35


In [6]:

# ------------------------------------------------------
# DISCRETIZATION SENSITIVITY — MAIN 5-VARIABLE SET
# ------------------------------------------------------

sensitivity_profile_discretization_main_set = discretization_diagnostic.copy()
sensitivity_profile_discretization_main_set["variable_set"] = MAIN_VARIABLE_SET
sensitivity_profile_discretization_main_set["sensitivity_type"] = "discretization_levels"
sensitivity_profile_discretization_main_set["main_result_flag"] = sensitivity_profile_discretization_main_set["levels"].eq(MAIN_LEVELS)

save_table(
    sensitivity_profile_discretization_main_set,
    "sensitivity_profile_discretization_main_set.csv",
    "Sensitivity profile discretization main set",
    "Sensitivity comparison of 3, 4, and 5 ordinal levels for the final 5-variable set.",
)

report_table_profile_sensitivity = sensitivity_profile_discretization_main_set[
    [
        "shock_id", "levels", "countries", "variables",
        "unique_profiles", "profile_uniqueness_ratio",
        "max_countries_per_profile", "is_main_choice",
    ]
].copy()

save_table(
    report_table_profile_sensitivity,
    "report_table_profile_sensitivity.csv",
    "Report table profile sensitivity",
    "Report-ready table for discretization sensitivity.",
)

display(sensitivity_profile_discretization_main_set)


Saved: sensitivity_profile_discretization_main_set.csv
Saved: report_table_profile_sensitivity.csv


,shock_id,levels,countries,variables,unique_profiles,profile_uniqueness_ratio,max_countries_per_profile,mean_countries_per_level_bin,is_main_choice,interpretation,variable_set,sensitivity_type,main_result_flag
0,2007,3,25,5,22,0.8800,2,8.3333,False,Sensitivity/diagnostic alternative.,baseline_5_variables,discretization_levels,False
1,2007,4,25,5,25,1.0000,1,6.2500,False,Sensitivity/diagnostic alternative.,baseline_5_variables,discretization_levels,False
2,2007,5,25,5,25,1.0000,1,5.0000,True,Main choice: five levels preserve ordinal deta...,baseline_5_variables,discretization_levels,True
3,2019,3,35,5,28,0.8000,2,11.6667,False,Sensitivity/diagnostic alternative.,baseline_5_variables,discretization_levels,False
4,2019,4,35,5,35,1.0000,1,8.7500,False,Sensitivity/diagnostic alternative.,baseline_5_variables,discretization_levels,False
5,2019,5,35,5,34,0.9714,2,7.0000,True,Main choice: five levels preserve ordinal deta...,baseline_5_variables,discretization_levels,True


In [7]:

# ------------------------------------------------------
# VARIABLE-SET SENSITIVITY
# ------------------------------------------------------

variable_set_frames = []

for set_name, score_cols in VARIABLE_SET_DEFINITIONS.items():
    for levels in [MAIN_LEVELS]:
        temp = compute_profile_poset_summary(
            country_df=structural,
            score_columns=score_cols,
            levels=levels,
            variable_set_name=set_name,
        )
        temp["sensitivity_type"] = "variable_set"
        temp["is_main_variable_set"] = set_name == MAIN_VARIABLE_SET
        variable_set_frames.append(temp)

sensitivity_variable_set_profile_summary = pd.concat(variable_set_frames, ignore_index=True)

save_table(
    sensitivity_variable_set_profile_summary,
    "sensitivity_variable_set_profile_summary.csv",
    "Sensitivity variable-set profile summary",
    "POSet quality metrics for the main five-variable set and a drop-energy sensitivity set.",
)

display(sensitivity_variable_set_profile_summary)


Saved: sensitivity_variable_set_profile_summary.csv


,shock_id,baseline_year,variable_set,n_variables,levels,countries,unique_profiles,profile_uniqueness_ratio,dominance_relations,hasse_edges,layers,frontier_profiles,frontier_countries,comparability_ratio,incomparability_ratio,sensitivity_type,is_main_variable_set
0,2007,2007,baseline_5_variables,5,5,25,25,1.0000,102,54,5,8,8,0.3400,0.6600,variable_set,True
1,2019,2019,baseline_5_variables,5,5,35,35,1.0000,146,75,5,12,12,0.2454,0.7546,variable_set,True
2,2007,2007,sensitivity_drop_energy_security,4,5,25,24,0.9600,122,46,6,6,6,0.4420,0.5580,variable_set,False
3,2019,2019,sensitivity_drop_energy_security,4,5,35,34,0.9714,222,83,7,3,3,0.3957,0.6043,variable_set,False


In [8]:

# ------------------------------------------------------
# EPSILON TOLERANCE AND EPSILON-MARGIN SENSITIVITY
# ------------------------------------------------------

sensitivity_epsilon_tolerance_main_set = epsilon_tolerance_summary.copy()
sensitivity_epsilon_tolerance_main_set["sensitivity_type"] = "epsilon_tolerance"
sensitivity_epsilon_tolerance_main_set["main_result_flag"] = False
sensitivity_epsilon_tolerance_main_set["reference_epsilon_flag"] = np.isclose(
    pd.to_numeric(sensitivity_epsilon_tolerance_main_set["epsilon"], errors="coerce"),
    REFERENCE_EPSILON,
)

# Add validity details if available.
if len(epsilon_tolerance_validity):
    validity_cols = [
        "shock_id", "baseline_year", "epsilon",
        "reciprocal_pairs", "cycle_detected", "is_valid_partial_order",
    ]
    validity_cols = [c for c in validity_cols if c in epsilon_tolerance_validity.columns]

    sensitivity_epsilon_tolerance_main_set = sensitivity_epsilon_tolerance_main_set.merge(
        epsilon_tolerance_validity[validity_cols],
        on=["shock_id", "baseline_year", "epsilon"],
        how="left",
        suffixes=("", "_validity"),
    )

sensitivity_epsilon_margin_main_set = epsilon_margin_summary.copy()
sensitivity_epsilon_margin_main_set["sensitivity_type"] = "epsilon_margin"
sensitivity_epsilon_margin_main_set["main_result_flag"] = False
sensitivity_epsilon_margin_main_set["reference_epsilon_flag"] = np.isclose(
    pd.to_numeric(sensitivity_epsilon_margin_main_set["epsilon_margin"], errors="coerce"),
    REFERENCE_EPSILON,
)

if len(epsilon_margin_validity):
    validity_cols = [
        "shock_id", "baseline_year", "epsilon_margin",
        "reciprocal_pairs", "cycle_detected", "is_valid_partial_order",
    ]
    validity_cols = [c for c in validity_cols if c in epsilon_margin_validity.columns]

    sensitivity_epsilon_margin_main_set = sensitivity_epsilon_margin_main_set.merge(
        epsilon_margin_validity[validity_cols],
        on=["shock_id", "baseline_year", "epsilon_margin"],
        how="left",
        suffixes=("", "_validity"),
    )

save_table(
    sensitivity_epsilon_tolerance_main_set,
    "sensitivity_epsilon_tolerance_main_set.csv",
    "Sensitivity epsilon tolerance main set",
    "Epsilon-tolerance sensitivity results from Notebook 09, including validity information.",
)

save_table(
    sensitivity_epsilon_margin_main_set,
    "sensitivity_epsilon_margin_main_set.csv",
    "Sensitivity epsilon margin main set",
    "Epsilon-margin robustness results from Notebook 10, including validity information.",
)

report_table_epsilon_margin = sensitivity_epsilon_margin_main_set[
    [
        "shock_id", "baseline_year", "epsilon_margin", "countries",
        "relations", "hasse_edges", "frontier_countries",
        "comparability_ratio", "incomparability_ratio", "is_valid_partial_order",
    ]
].copy()

save_table(
    report_table_epsilon_margin,
    "report_table_epsilon_margin.csv",
    "Report table epsilon margin",
    "Report-ready epsilon-margin robustness table.",
)

display(sensitivity_epsilon_tolerance_main_set)
display(sensitivity_epsilon_margin_main_set)


ValueError: You are trying to merge on object and int64 columns. If you wish to proceed you should use pd.concat

In [ ]:

# ------------------------------------------------------
# RECOVERY VALIDATION SENSITIVITY SUMMARY
# ------------------------------------------------------

sensitivity_recovery_validation_main_set = recovery_takeaways.copy()
sensitivity_recovery_validation_main_set["sensitivity_type"] = "recovery_validation"
sensitivity_recovery_validation_main_set["validation_role"] = "external_outcome_not_ordering"

save_table(
    sensitivity_recovery_validation_main_set,
    "sensitivity_recovery_validation_main_set.csv",
    "Sensitivity recovery validation main set",
    "Shock-level recovery validation takeaways for the main profile POSet.",
)

report_table_recovery_validation = sensitivity_recovery_validation_main_set[
    [
        "shock_id", "countries", "frontier_countries",
        "frontier_mean_minus_nonfrontier_mean_recovery",
        "frontier_median_minus_nonfrontier_median_recovery",
        "layer_recovery_spearman_correlation",
        "headline_takeaway",
    ]
].copy()

save_table(
    report_table_recovery_validation,
    "report_table_recovery_validation.csv",
    "Report table recovery validation",
    "Report-ready recovery validation table.",
)

display(sensitivity_recovery_validation_main_set)


In [ ]:

# ------------------------------------------------------
# ROBUSTNESS DASHBOARD AND SYNTHESIS
# ------------------------------------------------------

dashboard_rows = []

# Discretization synthesis.
for shock_id, group in sensitivity_profile_discretization_main_set.groupby("shock_id"):
    main = group[group["levels"].eq(MAIN_LEVELS)]
    if len(main):
        main_unique_ratio = main["profile_uniqueness_ratio"].iloc[0]
        main_unique_profiles = main["unique_profiles"].iloc[0]
    else:
        main_unique_ratio = np.nan
        main_unique_profiles = np.nan

    dashboard_rows.append({
        "sensitivity_area": "discretization",
        "shock_id": shock_id,
        "headline_metric": "profile_uniqueness_ratio_at_5_levels",
        "value": main_unique_ratio,
        "secondary_value": main_unique_profiles,
        "interpretation": "Five-level profiles preserve most country-level structural differentiation.",
        "risk_level": "low",
    })

# Variable set synthesis.
for _, row in sensitivity_variable_set_profile_summary.iterrows():
    if row["variable_set"] == MAIN_VARIABLE_SET:
        continue
    dashboard_rows.append({
        "sensitivity_area": "variable_set",
        "shock_id": row["shock_id"],
        "headline_metric": f"{row['variable_set']}_frontier_countries",
        "value": row["frontier_countries"],
        "secondary_value": row["comparability_ratio"],
        "interpretation": "Alternative variable set used as sensitivity only; compare directionally with main set.",
        "risk_level": "moderate",
    })

# Epsilon tolerance validity synthesis.
for shock_id, group in sensitivity_epsilon_tolerance_main_set.groupby("shock_id"):
    valid_count = int(group["is_valid_partial_order"].sum()) if "is_valid_partial_order" in group.columns else np.nan
    total_count = len(group)

    dashboard_rows.append({
        "sensitivity_area": "epsilon_tolerance",
        "shock_id": shock_id,
        "headline_metric": "valid_epsilon_values",
        "value": valid_count,
        "secondary_value": total_count,
        "interpretation": "Tolerance is diagnostic only because higher epsilon values can create invalid partial orders.",
        "risk_level": "high" if valid_count < total_count else "low",
    })

# Epsilon-margin validity synthesis.
for shock_id, group in sensitivity_epsilon_margin_main_set.groupby("shock_id"):
    valid_count = int(group["is_valid_partial_order"].sum()) if "is_valid_partial_order" in group.columns else np.nan
    total_count = len(group)

    dashboard_rows.append({
        "sensitivity_area": "epsilon_margin",
        "shock_id": shock_id,
        "headline_metric": "valid_epsilon_margin_values",
        "value": valid_count,
        "secondary_value": total_count,
        "interpretation": "Margin rule remains a valid conservative robustness check.",
        "risk_level": "low" if valid_count == total_count else "moderate",
    })

# Recovery validation synthesis.
for _, row in sensitivity_recovery_validation_main_set.iterrows():
    dashboard_rows.append({
        "sensitivity_area": "recovery_validation",
        "shock_id": row["shock_id"],
        "headline_metric": "frontier_mean_minus_nonfrontier_recovery",
        "value": row["frontier_mean_minus_nonfrontier_mean_recovery"],
        "secondary_value": row["layer_recovery_spearman_correlation"],
        "interpretation": row["headline_takeaway"],
        "risk_level": "interpretive_only",
    })

robustness_dashboard_table = pd.DataFrame(dashboard_rows)

synthesis_rows = [
    {
        "finding_area": "Main POSet structure",
        "finding": "The main 5-variable profile POSet produces substantial incomparability, supporting POSet rather than scalar ranking.",
        "supporting_output": "profile_poset_quality_diagnostics.csv",
        "report_use": "Results/methodology",
    },
    {
        "finding_area": "Discretization",
        "finding": "Five levels are defensible because they preserve profile differentiation while remaining interpretable; 3 and 4 levels are retained as sensitivity checks.",
        "supporting_output": "sensitivity_profile_discretization_main_set.csv",
        "report_use": "Sensitivity analysis",
    },
    {
        "finding_area": "Variable selection",
        "finding": "The final five variables remain conceptually distinct and prior correlation checks did not flag high redundancy.",
        "supporting_output": "candidate_variable_scorecard.csv; sensitivity_variable_set_profile_summary.csv",
        "report_use": "Data/methodology",
    },
    {
        "finding_area": "Epsilon tolerance",
        "finding": "Tolerance-based epsilon is informative but can violate partial-order validity at higher epsilon values.",
        "supporting_output": "sensitivity_epsilon_tolerance_main_set.csv",
        "report_use": "Sensitivity caveat",
    },
    {
        "finding_area": "Epsilon-margin",
        "finding": "The epsilon-margin rule is a cleaner conservative robustness check and remains valid across the tested grid.",
        "supporting_output": "sensitivity_epsilon_margin_main_set.csv",
        "report_use": "Robustness",
    },
    {
        "finding_area": "Recovery validation",
        "finding": "Frontier countries recover faster on average in both shocks, but the evidence is descriptive and not causal.",
        "supporting_output": "sensitivity_recovery_validation_main_set.csv",
        "report_use": "Validation/discussion",
    },
]

sensitivity_synthesis = pd.DataFrame(synthesis_rows)

save_table(
    robustness_dashboard_table,
    "robustness_dashboard_table.csv",
    "Robustness dashboard table",
    "Compact sensitivity dashboard across discretization, variable set, epsilon, and validation checks.",
)

save_table(
    sensitivity_synthesis,
    "sensitivity_synthesis.csv",
    "Sensitivity synthesis",
    "High-level synthesis of sensitivity findings and report-use guidance.",
)

display(robustness_dashboard_table)
display(sensitivity_synthesis)


In [ ]:

# ------------------------------------------------------
# REPORT-READY SENSITIVITY PARAGRAPHS
# ------------------------------------------------------

report_ready_sensitivity_paragraphs = pd.DataFrame([
    {
        "topic": "Sensitivity analysis overview",
        "report_text": (
            "Sensitivity analysis was conducted across discretization choices, variable-set alternatives, epsilon-tolerance "
            "relations, epsilon-margin robustness, and recovery validation. The purpose is not to replace the main "
            "five-variable profile POSet, but to check whether the conclusions are fragile to defensible methodological choices."
        ),
    },
    {
        "topic": "Discretization sensitivity",
        "report_text": (
            "The main analysis uses five ordinal levels because this preserves structural detail while remaining interpretable. "
            "Alternative 3- and 4-level discretizations were computed to assess whether the POSet structure depends excessively "
            "on the chosen number of levels."
        ),
    },
    {
        "topic": "Epsilon-tolerance sensitivity",
        "report_text": (
            "The epsilon-tolerance rule was tested over 0.00, 0.05, 0.10, 0.15, and 0.20 on the normalized 0–1 scale. "
            "Because tolerance can create reciprocal dominance and cycles, results from invalid epsilon values are treated "
            "only as diagnostics and not as valid POSet structures."
        ),
    },
    {
        "topic": "Epsilon-margin robustness",
        "report_text": (
            "The epsilon-margin rule was tested using the same grid but interpreted epsilon as a required minimum advantage. "
            "This rule is more conservative because it does not relax componentwise dominance. Epsilon = 0.10 is retained as "
            "a moderate reference robustness value."
        ),
    },
    {
        "topic": "Recovery validation sensitivity",
        "report_text": (
            "Recovery validation compares the pre-shock structural POSet with post-shock GDP recovery outcomes. Because recovery "
            "is not used in the ordering relation, this validation avoids outcome leakage. Results are interpreted descriptively, "
            "not causally."
        ),
    },
    {
        "topic": "Final sensitivity conclusion",
        "report_text": (
            "Overall, the sensitivity checks support presenting the five-variable, five-level profile POSet as the main empirical "
            "structure, with epsilon-margin and recovery validation used as robustness and interpretation layers rather than "
            "alternative main rankings."
        ),
    },
])

methodological_decision_candidates = pd.DataFrame([
    {
        "decision": "Report 5-level profile POSet as main result",
        "status": "final",
        "justification": "Balances interpretability, profile differentiation, and POSet structure.",
        "sensitivity_evidence": "3/4/5-level discretization diagnostic.",
    },
    {
        "decision": "Report epsilon-margin as robustness, not main result",
        "status": "final",
        "justification": "Margin rule remains valid but is country-level and secondary to profile POSet.",
        "sensitivity_evidence": "epsilon_margin_run_summary.csv.",
    },
    {
        "decision": "Treat epsilon-tolerance cautiously",
        "status": "final",
        "justification": "Tolerance can create invalid partial orders at higher epsilon values.",
        "sensitivity_evidence": "epsilon_validity_diagnostics.csv.",
    },
    {
        "decision": "Use recovery validation descriptively",
        "status": "final",
        "justification": "Recovery outcomes validate/interpret but do not construct the POSet.",
        "sensitivity_evidence": "recovery_validation_takeaway_table.csv.",
    },
])

save_table(
    report_ready_sensitivity_paragraphs,
    "report_ready_sensitivity_paragraphs.csv",
    "Report-ready sensitivity paragraphs",
    "Report-ready text for the sensitivity analysis section.",
)

save_table(
    methodological_decision_candidates,
    "methodological_decision_candidates.csv",
    "Methodological decision candidates",
    "Final methodological decisions supported by sensitivity analysis.",
)

display(report_ready_sensitivity_paragraphs)
display(methodological_decision_candidates)


In [ ]:

# ------------------------------------------------------
# AUDIT WORKBOOK
# ------------------------------------------------------

audit_xlsx = AUDIT_DIR / "12_Sensitivity_Analysis_Audit.xlsx"

with pd.ExcelWriter(audit_xlsx, engine="xlsxwriter") as writer:
    sensitivity_synthesis.to_excel(writer, sheet_name="synthesis", index=False)
    robustness_dashboard_table.to_excel(writer, sheet_name="dashboard", index=False)
    sensitivity_profile_discretization_main_set.to_excel(writer, sheet_name="discretization", index=False)
    sensitivity_variable_set_profile_summary.to_excel(writer, sheet_name="variable_sets", index=False)
    sensitivity_epsilon_tolerance_main_set.to_excel(writer, sheet_name="epsilon_tolerance", index=False)
    sensitivity_epsilon_margin_main_set.to_excel(writer, sheet_name="epsilon_margin", index=False)
    sensitivity_recovery_validation_main_set.to_excel(writer, sheet_name="recovery_validation", index=False)
    report_table_profile_sensitivity.to_excel(writer, sheet_name="report_profile", index=False)
    report_table_epsilon_margin.to_excel(writer, sheet_name="report_epsilon_margin", index=False)
    report_table_recovery_validation.to_excel(writer, sheet_name="report_recovery", index=False)
    report_ready_sensitivity_paragraphs.to_excel(writer, sheet_name="report_paragraphs", index=False)
    methodological_decision_candidates.to_excel(writer, sheet_name="decision_candidates", index=False)
    pd.DataFrame(table_inventory_rows).to_excel(writer, sheet_name="table_inventory", index=False)

print("Audit workbook saved:", audit_xlsx.resolve())


In [ ]:

# ------------------------------------------------------
# FINAL COMPLETION CHECK
# ------------------------------------------------------

table_inventory = pd.DataFrame(table_inventory_rows)
table_inventory.to_csv(PROCESSED_DIR / "sensitivity_analysis_table_inventory.csv", index=False)
table_inventory.to_csv(DIAGNOSTICS_DIR / "sensitivity_analysis_table_inventory.csv", index=False)

expected_outputs = [
    "sensitivity_profile_discretization_main_set.csv",
    "sensitivity_variable_set_profile_summary.csv",
    "sensitivity_epsilon_tolerance_main_set.csv",
    "sensitivity_epsilon_margin_main_set.csv",
    "sensitivity_recovery_validation_main_set.csv",
    "robustness_dashboard_table.csv",
    "sensitivity_synthesis.csv",
    "report_table_profile_sensitivity.csv",
    "report_table_epsilon_margin.csv",
    "report_table_recovery_validation.csv",
    "report_ready_sensitivity_paragraphs.csv",
    "methodological_decision_candidates.csv",
]

output_check = pd.DataFrame([
    {
        "file_name": f,
        "processed_exists": (PROCESSED_DIR / f).exists(),
        "diagnostics_exists": (DIAGNOSTICS_DIR / f).exists(),
        "tables_exists": (TABLES_DIR / f).exists(),
    }
    for f in expected_outputs
])

output_check.to_csv(DIAGNOSTICS_DIR / "output_check.csv", index=False)

print("12 SENSITIVITY ANALYSIS COMPLETE")
print("=" * 90)

display(output_check)

print("\nKey outputs to inspect/send:")
print("- 12_Sensitivity_Analysis_Audit.xlsx")
print("- sensitivity_synthesis.csv")
print("- robustness_dashboard_table.csv")
print("- report_ready_sensitivity_paragraphs.csv")

print("\nNext notebook:")
print("13_Final_Visuals_2002_5Var.ipynb")
